# Perfil da bronze

**Para que serve:** conhecer a base antes de limpar (item 2 do guia). Este notebook **só lê** a bronze, sem gravar nada. Cada seção termina com a sua conclusão e a regra proposta para a silver. Isso alimenta as decisões S3 em `docs/decisoes_arquitetura_lakehouse.md`.

**Como usar:** rode uma célula por vez (**Shift + Enter**), leia o resultado e preencha o "✍️ Sua conclusão" logo abaixo (dois cliques na célula para editar).

## 0. Preparar
Abre as tabelas da bronze **pelo `_delta_log`** (regra F3) e cria *views* só com a **carga mais recente**. Assim o resto do notebook não precisa se preocupar com cargas antigas.

In [1]:
from pathlib import Path
import duckdb
from deltalake import DeltaTable

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
con = duckdb.connect()
con.execute("SET memory_limit='1GB'; SET threads=2")

def abrir_bronze(tabela, nome):
    ds = DeltaTable(str(RAIZ / "lake" / "bronze" / tabela)).to_pyarrow_dataset()
    con.register(f"_{nome}_ds", ds)
    con.execute(f"""CREATE OR REPLACE VIEW {nome} AS
                   SELECT * FROM _{nome}_ds
                   WHERE _data_carga = (SELECT MAX(_data_carga) FROM _{nome}_ds)""")

abrir_bronze("transaction_data", "vendas")
abrir_bronze("product", "produtos")
abrir_bronze("hh_demographic", "demografia")

con.sql("SELECT 'vendas' AS tabela, COUNT(*) AS linhas FROM vendas UNION ALL SELECT 'produtos', COUNT(*) FROM produtos UNION ALL SELECT 'demografia', COUNT(*) FROM demografia").show()

┌────────────┬─────────┐
│   tabela   │ linhas  │
│  varchar   │  int64  │
├────────────┼─────────┤
│ vendas     │ 2595732 │
│ produtos   │   92353 │
│ demografia │     801 │
└────────────┴─────────┘



Esperado: vendas 2.595.732 · produtos 92.353 · demografia 801.

## 1. Cobertura
**Pergunta:** o que tem na base? Quantos clientes, cestas, produtos, lojas e qual o período.
Esperado (gabarito): 2.500 clientes, 582 lojas, dias de 1 a 711, semanas de 1 a 102.

In [2]:
con.sql("""
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT household_key) AS clientes,
       COUNT(DISTINCT BASKET_ID) AS cestas,
       COUNT(DISTINCT PRODUCT_ID) AS produtos,
       COUNT(DISTINCT STORE_ID) AS lojas,
       MIN(CAST(DAY AS INT)) AS dia_min, MAX(CAST(DAY AS INT)) AS dia_max,
       MIN(CAST(WEEK_NO AS INT)) AS semana_min, MAX(CAST(WEEK_NO AS INT)) AS semana_max
FROM vendas
""").show(max_width=10000)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────┬────────┬──────────┬───────┬─────────┬─────────┬────────────┬────────────┐
│ linhas  │ clientes │ cestas │ produtos │ lojas │ dia_min │ dia_max │ semana_min │ semana_max │
│  int64  │  int64   │ int64  │  int64   │ int64 │  int32  │  int32  │   int32    │   int32    │
├─────────┼──────────┼────────┼──────────┼───────┼─────────┼─────────┼────────────┼────────────┤
│ 2595732 │     2500 │ 276484 │    92339 │   582 │       1 │     711 │          1 │        102 │
└─────────┴──────────┴────────┴──────────┴───────┴─────────┴─────────┴────────────┴────────────┘



**✍️ Conclusão:**

- Cobertura bate com o gabarito: **2.500** clientes, **582** lojas, dias **1 a 711**, semanas **1 a 102**.
- **276.484** cestas e **92.339** produtos vendidos. O cadastro tem 92.353, então **14 produtos nunca foram vendidos** (normal).

**Regra para a silver:** nenhuma. Período e chaves completos.

## 2. O texto vira número?
**Pergunta:** na bronze tudo é texto. Algum valor **não converte** para número? (`TRY_CAST` devolve nulo quando não consegue.)
Esperado: tudo 0. Se aparecer algo, vira regra de rejeição na silver.

In [4]:
con.sql("""
SELECT
  COUNT(*) FILTER (WHERE TRY_CAST(SALES_VALUE AS DECIMAL(18,4)) IS NULL) AS sales_value_invalido,
  COUNT(*) FILTER (WHERE TRY_CAST(QUANTITY AS INT) IS NULL) AS quantity_invalido,
  COUNT(*) FILTER (WHERE TRY_CAST(RETAIL_DISC AS DECIMAL(18,4)) IS NULL) AS retail_disc_invalido,
  COUNT(*) FILTER (WHERE TRY_CAST(COUPON_DISC AS DECIMAL(18,4)) IS NULL) AS coupon_disc_invalido,
  COUNT(*) FILTER (WHERE TRY_CAST(COUPON_MATCH_DISC AS DECIMAL(18,4)) IS NULL) AS coupon_match_invalido,
  COUNT(*) FILTER (WHERE TRY_CAST(BASKET_ID AS BIGINT) IS NULL) AS basket_id_invalido,
  COUNT(*) FILTER (WHERE TRY_CAST(DAY AS INT) IS NULL) AS day_invalido,
  COUNT(*) FILTER (WHERE TRY_CAST(TRANS_TIME AS INT) IS NULL) AS trans_time_invalido
FROM vendas
""").show(max_width=10000)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┬───────────────────┬──────────────────────┬──────────────────────┬───────────────────────┬────────────────────┬──────────────┬─────────────────────┐
│ sales_value_invalido │ quantity_invalido │ retail_disc_invalido │ coupon_disc_invalido │ coupon_match_invalido │ basket_id_invalido │ day_invalido │ trans_time_invalido │
│        int64         │       int64       │        int64         │        int64         │         int64         │       int64        │    int64     │        int64        │
├──────────────────────┼───────────────────┼──────────────────────┼──────────────────────┼───────────────────────┼────────────────────┼──────────────┼─────────────────────┤
│                    0 │                 0 │                    0 │                    0 │                     0 │                  0 │            0 │                   0 │
└──────────────────────┴───────────────────┴──────────────────────┴──────────────────────┴───────────────────────┴────────────────────┴

**✍️ Conclusão:**

- **0** valores inválidos nas 8 colunas numéricas. Todo texto converte para número.

**Regra para a silver:** converter com `CAST` e manter o **teste de conversão** como verificação automática (pega cargas futuras com problema).

## 3. O grão da cesta
**Pergunta:** cada `BASKET_ID` tem **um cliente, uma loja e um dia só**? Se não tiver, o ticket médio e os itens por cesta ficam errados.
Esperado: 0.

In [5]:
con.sql("""
SELECT COUNT(*) AS cestas_com_problema FROM (
  SELECT BASKET_ID FROM vendas GROUP BY BASKET_ID
  HAVING COUNT(DISTINCT household_key) > 1
      OR COUNT(DISTINCT STORE_ID) > 1
      OR COUNT(DISTINCT DAY) > 1)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┐
│ cestas_com_problema │
│        int64        │
├─────────────────────┤
│                   0 │
└─────────────────────┘



**✍️ Conclusão:**

- **0** cestas com mais de um cliente, loja ou dia. O grão da cesta é confiável.

**Regra para a silver:** `basket_id` é a chave da cesta; ticket médio e itens por cesta podem ser calculados direto. Vira **teste automático**.

## 4. Valores estranhos
**Pergunta:** quantas linhas têm valor negativo, quantidade zero, itens grátis, descontos com sinal trocado?
Daqui saem as regras de `sales_value`, `quantity` e descontos. Gabarito do RAG: quantidade zero = 14.466; `RETAIL_DISC` positivo = 2. Os números aqui ainda incluem combustível e pseudo-itens, então alguns podem ser maiores. Anote os dois.

In [6]:
con.sql("""
WITH v AS (SELECT CAST(SALES_VALUE AS DECIMAL(18,4)) s, CAST(QUANTITY AS INT) q,
                  CAST(RETAIL_DISC AS DECIMAL(18,4)) rd, CAST(COUPON_DISC AS DECIMAL(18,4)) cd,
                  CAST(COUPON_MATCH_DISC AS DECIMAL(18,4)) cm FROM vendas)
SELECT
  COUNT(*) FILTER (WHERE s < 0) AS valor_negativo,
  COUNT(*) FILTER (WHERE s = 0) AS valor_zero,
  COUNT(*) FILTER (WHERE s = 0 AND q > 0) AS itens_gratis,
  COUNT(*) FILTER (WHERE q = 0) AS quantidade_zero,
  ROUND(SUM(s) FILTER (WHERE q = 0), 2) AS vendas_com_quantidade_zero,
  COUNT(*) FILTER (WHERE q < 0) AS quantidade_negativa,
  COUNT(*) FILTER (WHERE q > 100) AS quantidade_acima_100,
  COUNT(*) FILTER (WHERE rd > 0) AS retail_disc_positivo,
  COUNT(*) FILTER (WHERE cd > 0) AS coupon_disc_positivo,
  COUNT(*) FILTER (WHERE cm > 0) AS coupon_match_positivo
FROM v
""").show(max_width=10000)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────┬──────────────┬─────────────────┬────────────────────────────┬─────────────────────┬──────────────────────┬──────────────────────┬──────────────────────┬───────────────────────┐
│ valor_negativo │ valor_zero │ itens_gratis │ quantidade_zero │ vendas_com_quantidade_zero │ quantidade_negativa │ quantidade_acima_100 │ retail_disc_positivo │ coupon_disc_positivo │ coupon_match_positivo │
│     int64      │   int64    │    int64     │      int64      │       decimal(38,2)        │        int64        │        int64         │        int64         │        int64         │         int64         │
├────────────────┼────────────┼──────────────┼─────────────────┼────────────────────────────┼─────────────────────┼──────────────────────┼──────────────────────┼──────────────────────┼───────────────────────┤
│              0 │      18879 │         4451 │           14466 │                      19.28 │                   0 │                23136 │                   10 │   

**✍️ Conclusão:**

*(números antes de tirar a não mercadoria)*
- Valor negativo: **0** · quantidade negativa: **0**. Devolução não aparece como sinal negativo.
- Quantidade zero: **14.466** linhas (✅ gabarito), somando só **US$ 19,28**. Conta: valor zero 18.879 = 14.466 + 4.451 − **38** linhas de quantidade zero com algum valor. Na prática, registro vazio, não venda.
- Itens grátis (valor 0 com quantidade > 0): **4.451** (gabarito 3.994 já sem não mercadoria).
- Quantidade acima de 100: **23.136**. A seção 6 mostra a causa: combustível e `MISC SALES TRAN` têm quantidade máxima de ~85 a 90 mil (unidade de medida diferente).
- `RETAIL_DISC` positivo: **10** (gabarito 2). Hipótese: os outros 8 estão na não mercadoria. `COUPON_DISC` e `COUPON_MATCH_DISC` positivos: **0**.

**Regras propostas:**
- `quantity = 0` → **rejeitar** (motivo "quantidade zero"): 14.466 linhas, US$ 19,28.
- Item grátis → **manter** e marcar `is_free_item`.
- `retail_disc > 0` em mercadoria → **manter** e marcar como estorno (como no RAG), e conferir na silver se sobram 2.
- Quantidade alta → resolvida pela marcação de não mercadoria (seção 6).

## 5. O horário (`TRANS_TIME`)
**Pergunta:** é um horário válido (0000 a 2359, minuto até 59)? Todos têm o mesmo tamanho de texto (ex.: `0930` e `930`)?

In [7]:
con.sql("""
SELECT MIN(CAST(TRANS_TIME AS INT)) AS minimo, MAX(CAST(TRANS_TIME AS INT)) AS maximo,
       COUNT(*) FILTER (WHERE CAST(TRANS_TIME AS INT) % 100 > 59) AS minuto_invalido,
       COUNT(*) FILTER (WHERE CAST(TRANS_TIME AS INT) > 2359) AS hora_invalida
FROM vendas
""").show(max_width=10000)
con.sql("SELECT LENGTH(TRANS_TIME) AS tamanho, COUNT(*) AS linhas FROM vendas GROUP BY 1 ORDER BY 1").show()

┌────────┬────────┬─────────────────┬───────────────┐
│ minimo │ maximo │ minuto_invalido │ hora_invalida │
│ int32  │ int32  │      int64      │     int64     │
├────────┼────────┼─────────────────┼───────────────┤
│      0 │   2359 │               0 │             0 │
└────────┴────────┴─────────────────┴───────────────┘

┌─────────┬─────────┐
│ tamanho │ linhas  │
│  int64  │  int64  │
├─────────┼─────────┤
│       4 │ 2595732 │
└─────────┴─────────┘



**✍️ Conclusão:**

- Horário entre **0000 e 2359**, **0** minutos ou horas inválidos, e todas as **2.595.732** linhas com 4 caracteres.

**Regra para a silver:** converter para horário (`trans_time` como `TIME`, ex.: `0930` → 09:30) e derivar `trans_hour` para análises por hora. Manter o teste de horário válido.

## 6. Departamentos e faturamento
**Pergunta:** quais departamentos existem e quanto cada um fatura? Serve para revisar a lista de **não mercadoria** (combustível `KIOSK-GAS`, `MISC SALES TRAN`, `MISC. TRANS.`, `COUP/STR & MFG`) e ver se ficou algum parecido de fora.
Gabarito: não mercadoria = 32.805 linhas, US$ 681.121,53.

In [8]:
con.sql("""
SELECT p.DEPARTMENT, COUNT(*) AS linhas,
       ROUND(SUM(CAST(v.SALES_VALUE AS DECIMAL(18,4))), 2) AS vendas,
       MAX(CAST(v.QUANTITY AS INT)) AS maior_quantidade
FROM vendas v JOIN produtos p USING (PRODUCT_ID)
GROUP BY 1 ORDER BY vendas DESC
""").show(max_rows=60)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬───────────────┬──────────────────┐
│   DEPARTMENT    │ linhas  │    vendas     │ maior_quantidade │
│     varchar     │  int64  │ decimal(38,2) │      int32       │
├─────────────────┼─────────┼───────────────┼──────────────────┤
│ GROCERY         │ 1646076 │    4093814.14 │               60 │
│ DRUG GM         │  277232 │    1055358.03 │               90 │
│ PRODUCE         │  257290 │     557452.11 │              144 │
│ MEAT            │   88416 │     548786.81 │               40 │
│ KIOSK-GAS       │   22059 │     544222.28 │            89638 │
│ MEAT-PCKGD      │  111957 │     412436.77 │               20 │
│ DELI            │   62787 │     260866.51 │               15 │
│ PASTRY          │   38179 │     121739.86 │               32 │
│ MISC SALES TRAN │    6050 │     119960.04 │            85055 │
│ NUTRITION       │   32164 │      97669.04 │               30 │
│ SEAFOOD-PCKGD   │   11216 │      63093.62 │               15 │
│ FLORAL          │    45

**✍️ Conclusão:**

- **Não mercadoria pelos departamentos:** KIOSK-GAS (22.059 linhas · US$ 544.222,28), MISC SALES TRAN (6.050 · US$ 119.960,04), MISC. TRANS. (2.351 · US$ 10.678,69), COUP/STR & MFG (817 · US$ 1.027,72) = **31.277 linhas · US$ 675.888,73**.
- O gabarito dá **32.805 · US$ 681.121,53**. A diferença (**1.528 linhas · US$ 5.232,80**) é a commodity `COUPON/MISC ITEMS`, que está espalhada em outros departamentos. Por isso a regra também olha a commodity.
- **Quantidade absurda só na não mercadoria:** KIOSK-GAS chega a 89.638 e MISC SALES TRAN a 85.055. Na mercadoria o máximo é 144 (PRODUCE, plausível para granel).
- **Departamento vazio:** **7.839** linhas com US$ 0,00 e quantidade máxima 0. São todas linhas de quantidade zero, já cobertas pela regra de rejeição.
- Há vários departamentos minúsculos (RX, VIDEO, POSTAL CENTER, CHARITABLE CONT…), com menos de US$ 200 cada. São mercadoria real, mas irrelevantes.

**Regras propostas:**
- **Manter** a não mercadoria na silver com `is_merchandise = false` (decisão A1), por departamento (as 4 acima) **ou** commodity `COUPON/MISC ITEMS`. Conferir se dá exatamente 32.805 linhas.
- Painel e métricas da gold usam só `is_merchandise = true`.
- Departamentos minúsculos: manter como estão.

## 7. Cadastro de produtos
**Pergunta:** a chave `PRODUCT_ID` é única? Quantos campos vazios? Quais marcas existem?

In [9]:
con.sql("""
SELECT COUNT(*) AS linhas, COUNT(DISTINCT PRODUCT_ID) AS produtos_distintos,
       COUNT(*) FILTER (WHERE DEPARTMENT IS NULL OR TRIM(DEPARTMENT) = '') AS departamento_vazio,
       COUNT(*) FILTER (WHERE COMMODITY_DESC IS NULL OR TRIM(COMMODITY_DESC) = '') AS commodity_vazio,
       COUNT(*) FILTER (WHERE CURR_SIZE_OF_PRODUCT IS NULL OR TRIM(CURR_SIZE_OF_PRODUCT) = '') AS tamanho_vazio
FROM produtos
""").show()
con.sql("SELECT BRAND, COUNT(*) AS produtos FROM produtos GROUP BY 1 ORDER BY 2 DESC").show()
con.sql("SELECT COUNT(*) AS vendas_sem_cadastro FROM vendas v ANTI JOIN produtos p USING (PRODUCT_ID)").show()

┌────────┬────────────────────┬────────────────────┬─────────────────┬───────────────┐
│ linhas │ produtos_distintos │ departamento_vazio │ commodity_vazio │ tamanho_vazio │
│ int64  │       int64        │       int64        │      int64      │     int64     │
├────────┼────────────────────┼────────────────────┼─────────────────┼───────────────┤
│  92353 │              92353 │                 15 │              15 │         30607 │
└────────┴────────────────────┴────────────────────┴─────────────────┴───────────────┘

┌──────────┬──────────┐
│  BRAND   │ produtos │
│ varchar  │  int64   │
├──────────┼──────────┤
│ National │    78537 │
│ Private  │    13816 │
└──────────┴──────────┘

┌─────────────────────┐
│ vendas_sem_cadastro │
│        int64        │
├─────────────────────┤
│                   0 │
└─────────────────────┘



**✍️ Conclusão:**

- `PRODUCT_ID` único: **92.353** linhas = **92.353** distintos ✅. **0** vendas sem cadastro (sem órfãos).
- **15** produtos sem departamento e sem commodity (são os das 7.839 linhas de quantidade zero).
- Tamanho (`CURR_SIZE_OF_PRODUCT`) vazio em **30.607** produtos (33%).
- Marcas: **National 78.537** · **Private 13.816** (marca própria).

**Regras propostas:**
- Texto vazio vira **nulo** (`TRIM` e `NULLIF`). Tamanho vazio fica nulo, porque o painel não usa.
- Departamento vazio → `'UNKNOWN'` na dimensão de produto, para não perder o produto no join.
- `brand` mantém National/Private (alimenta "Marcas" no painel).

## 8. Demografia
**Pergunta:** quantos clientes têm demografia (gabarito: 801 de 2.500)? Aparecem valores como `Unknown` escritos como texto, que na prática são "sem informação"?

In [10]:
con.sql("SELECT COUNT(*) AS linhas, COUNT(DISTINCT household_key) AS clientes FROM demografia").show()
for coluna in ["AGE_DESC", "INCOME_DESC", "HOMEOWNER_DESC", "HH_COMP_DESC",
               "HOUSEHOLD_SIZE_DESC", "KID_CATEGORY_DESC", "MARITAL_STATUS_CODE"]:
    con.sql(f"SELECT {coluna}, COUNT(*) AS n FROM demografia GROUP BY 1 ORDER BY 2 DESC").show()

┌────────┬──────────┐
│ linhas │ clientes │
│ int64  │  int64   │
├────────┼──────────┤
│    801 │      801 │
└────────┴──────────┘

┌──────────┬───────┐
│ AGE_DESC │   n   │
│ varchar  │ int64 │
├──────────┼───────┤
│ 45-54    │   288 │
│ 35-44    │   194 │
│ 25-34    │   142 │
│ 65+      │    72 │
│ 55-64    │    59 │
│ 19-24    │    46 │
└──────────┴───────┘

┌─────────────┬───────┐
│ INCOME_DESC │   n   │
│   varchar   │ int64 │
├─────────────┼───────┤
│ 50-74K      │   192 │
│ 35-49K      │   172 │
│ 75-99K      │    96 │
│ 25-34K      │    77 │
│ 15-24K      │    74 │
│ Under 15K   │    61 │
│ 125-149K    │    38 │
│ 100-124K    │    34 │
│ 150-174K    │    30 │
│ 250K+       │    11 │
│ 175-199K    │    11 │
│ 200-249K    │     5 │
└─────────────┴───────┘
  12 rows   2 columns

┌─────────────────┬───────┐
│ HOMEOWNER_DESC  │   n   │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ Homeowner       │   504 │
│ Unknown         │   233 │
│ Renter          │    42 │
│ Probab

**✍️ Conclusão:**

- **801** linhas = **801** clientes ✅. Só **32%** dos 2.500 clientes têm demografia.
- "Sem informação" aparece de **4 jeitos diferentes**: `Unknown` em HOMEOWNER_DESC (233) e HH_COMP_DESC (73), `None/Unknown` em KID_CATEGORY_DESC (558) e `U` em MARITAL_STATUS_CODE (344).
- `None/Unknown` mistura "sem filhos" com "não sabemos", e não dá para separar.
- Faixas de idade e renda são texto (`45-54`, `50-74K`), com ordem natural que o texto não guarda.

**Regras propostas:**
- Padronizar os "sem informação" num valor só (`'Unknown'`) e nunca transformar em nulo silencioso.
- `marital_status_code`: traduzir `A`/`B`/`U` conforme o dicionário da Dunnhumby (confirmar o significado de A e B antes de rotular).
- Criar coluna de **ordem** para idade e renda (para gráficos ordenados).
- Os 1.699 clientes sem demografia entram na dimensão de cliente com demografia `'Unknown'`.

## 9. Duplicatas
**Pergunta:** existem linhas de venda **totalmente iguais**? Se sim, é erro ou compra repetida do mesmo item na mesma cesta?
Esperado (gabarito): 0.

In [11]:
con.sql("""
SELECT COUNT(*) AS grupos_duplicados FROM (
  SELECT household_key, BASKET_ID, DAY, PRODUCT_ID, QUANTITY, SALES_VALUE, STORE_ID,
         RETAIL_DISC, TRANS_TIME, WEEK_NO, COUPON_DISC, COUPON_MATCH_DISC
  FROM vendas GROUP BY ALL HAVING COUNT(*) > 1)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┐
│ grupos_duplicados │
│       int64       │
├───────────────────┤
│                 0 │
└───────────────────┘



**✍️ Conclusão:**

- **0** linhas totalmente iguais ✅.

**Regra para a silver:** nenhuma deduplicação. Manter o **teste de duplicata** como verificação automática.

## Resumo: regras propostas para a silver
Uma linha por regra: **o que**, **quantas linhas**, **o que fazer** (manter, marcar, rejeitar). *Propostas: viram decisão quando aprovadas no documento de decisões (S3).*

| # | Regra | Linhas afetadas | Ação |
|---|---|---|---|
| 1 | `quantity = 0` | 14.466 (US$ 19,28) | **rejeitar** → `_rejected`, motivo "quantidade zero" |
| 2 | Não mercadoria: 4 departamentos + commodity `COUPON/MISC ITEMS` | 32.805 (US$ 681.121,53) esperado | **manter** e marcar `is_merchandise = false` |
| 3 | Item grátis (valor 0, quantidade > 0) | 4.451 (3.994 em mercadoria) | **manter** e marcar `is_free_item` |
| 4 | `retail_disc > 0` | 10 (2 em mercadoria) | **manter** e marcar estorno |
| 5 | Texto → número (`CAST`) | 0 inválidos | converter; **teste automático** |
| 6 | Grão da cesta | 0 problemas | **teste automático** |
| 7 | `trans_time` → `TIME` + `trans_hour` | 0 inválidos | converter; **teste automático** |
| 8 | Texto vazio no produto | 15 sem departamento · 30.607 sem tamanho | vazio → nulo; departamento → `'UNKNOWN'` |
| 9 | "Sem informação" na demografia (`Unknown`, `None/Unknown`, `U`) | 233 · 73 · 558 · 344 | padronizar como `'Unknown'` |
| 10 | Clientes sem demografia | 1.699 de 2.500 | entram com demografia `'Unknown'` |
| 11 | Duplicatas | 0 | sem deduplicação; **teste automático** |

**Reconciliação esperada na silver:** 2.595.732 bronze = **2.581.266** limpas + **14.466** rejeitadas (0,56%, abaixo do limite de 1%).

**Pendências para conferir já na silver:** (a) não mercadoria = exatamente 32.805 linhas; (b) `retail_disc > 0` em mercadoria = 2; (c) significado de `A`/`B` no estado civil.